# 🥈 Silver Layer: Clean News and Extract Ticker-Specific Sentiment
**Purpose:** Parse the raw Alpha Vantage JSON, standardize timestamps, and use higher-order array functions to extract the precise sentiment and relevance scores for our specific target ticker.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# 1. Configuration
CATALOG = "portfolio"
SCHEMA = "market_data"
BRONZE_NEWS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_news_sentiment"
SILVER_NEWS_TABLE = f"{CATALOG}.{SCHEMA}.silver_news_sentiment"

In [0]:
# 2. Read and Cleanse
df_raw_news = spark.read.table(BRONZE_NEWS_TABLE)

df_silver_news = df_raw_news.withColumn("published_datetime", F.to_timestamp(F.col("time_published"), "yyyyMMdd'T'HHmmss")) \
                            .withColumn("Date", F.to_date(F.col("published_datetime"))) 

# Use Spark's 'filter' expression to isolate the specific ticker object from the array
df_silver_news = df_silver_news.withColumn(
    "target_ticker_obj", 
    F.expr("filter(ticker_sentiment, x -> x.ticker == ticker_symbol)[0]")
)

# Extract the scores and cast them to doubles for downstream math
df_silver_news = df_silver_news.withColumn("sentiment_score", F.col("target_ticker_obj.ticker_sentiment_score").cast("double")) \
                               .withColumn("relevance_score", F.col("target_ticker_obj.relevance_score").cast("double")) \
                               .select("ticker_symbol", "Date", "title", "url", "relevance_score", "sentiment_score") \
                               .dropDuplicates(["url"]) \
                               .dropna(subset=["sentiment_score", "Date"])

In [0]:
df_silver_news.display()

In [0]:
# 3. Upsert into Silver
if spark.catalog.tableExists(SILVER_NEWS_TABLE):
    delta_table = DeltaTable.forName(spark, SILVER_NEWS_TABLE)
    
    delta_table.alias("target").merge(
        df_silver_news.alias("source"),
        "target.url = source.url"  # merge on URL
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    df_silver_news.write.format("delta").mode("overwrite").saveAsTable(SILVER_NEWS_TABLE)

display(spark.read.table(SILVER_NEWS_TABLE).orderBy(F.col("Date").desc()).limit(5))